### 02 — FRED Macro Data Quality
Cel: ocena kompletności, poprawności i przydatności analitycznej danych makroekonomicznych.
Tabele: silver.fred_macro_indicators, silver.fred_macro_metadata_indicators, gold.fred_with_dimension, gold.macro_impact_on_tech

In [0]:
%sql
DESCRIBE silver.fred_macro_indicators

In [0]:
%sql
SELECT 
  indicator_id,
  COUNT(*) AS total_rows,
  MIN(date) AS min_date,
  max(date) AS max_date,
  COUNT(*) - COUNT(value) AS null_value_count,
  ROUND((COUNT(*) - COUNT(value)) * 100.0 / COUNT(*), 2) AS null_percent
FROM silver.fred_macro_indicators
GROUP BY indicator_id
ORDER BY indicator_id
-- com
/* 4% nulli w dgs10, dgs2 i t10yie to weekendy i swieta, nie wymaga zadnych akcji */

In [0]:
%sql
SELECT 
  indicator_id,
  MONTHS_BETWEEN(MAX(date), MIN(date)) + 1 AS expected_months,
  COUNT(DISTINCT DATE_TRUNC('month', date)) AS actual_months,
  MONTHS_BETWEEN(MAX(date), MIN(date)) + 1 - COUNT(DISTINCT DATE_TRUNC('month', date)) AS missing_months
FROM silver.fred_macro_indicators
WHERE indicator_id IN('cpiaucsl', 'cpilfesl', 'fedfunds', 'unrate', 'indpro')
GROUP BY indicator_id
ORDER BY indicator_id

/* wszystkie serie miesieczne mają pełną ciągłość */

In [0]:
%sql
SELECT f.indicator_id,
COUNT(DISTINCT m.indicator_id) AS has_metadata
FROM silver.fred_macro_indicators f
LEFT JOIN silver.fred_macro_metadata_indicators m
ON f.indicator_id = m.indicator_id
GROUP BY f.indicator_id

In [0]:
%sql
SELECT indicator_id,
  MIN(value) AS min_value,
  MAX(value) AS max_value,
  ROUND(AVG(value), 2) AS avg_value
FROM silver.fred_macro_indicators
GROUP BY indicator_id

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.macro_impact_on_tech
/* total rows 13 to zbyt mala probka, aby wykonac sensowne analizy, powinno zmienic sie po pelnym bootstrap. */

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  COUNT(monthly_return) AS has_return,
  COUNT(cpiaucsl) AS has_cpi,
  COUNT(fedfunds) AS has_fedfunds,
  COUNT(dgs10) AS has_dgs10,
  COUNT(unrate) AS has_unrate
FROM gold.macro_impact_on_tech

In [0]:
%sql
SELECT year_month, cpiaucsl, fedfunds, unrate
FROM gold.macro_impact_on_tech
WHERE cpiaucsl IS NULL 
   OR fedfunds IS NULL 
   OR unrate IS NULL
ORDER BY year_month

/* brak danych miesiecznych 

In [0]:
%sql
SELECT indicator_id, date, value
FROM silver.fred_macro_indicators
WHERE indicator_id IN ('cpiaucsl', 'unrate')
  AND date BETWEEN '2025-09-01' AND '2025-11-30'
ORDER BY indicator_id, date

### Wnioski fred
1. Nulle ~4% w seriach dziennych (DGS10, DGS2, T10YIE) to weekendy i święta - expected, nie wymaga akcji.
2. Serie miesięczne (CPI, CPILFESL, FEDFUNDS, UNRATE, INDPRO) mają pełną ciągłość - zero brakujących miesięcy.
3. 100% pokrycie metadata dla wszystkich 8 wskaźników - join w gold nie traci rekordów.
4. Zakresy wartości wszystkich wskaźników zgodne z oczekiwaniami domenowymi - brak outlierów, brak nieprawidłowych wartości ujemnych.
5. gold.macro_impact_on_tech: 13 wierszy (marzec 2025 - marzec 2026) - zbyt mała próbka na wiarygodne wnioski makro, rozważ rozszerzenie historii OHLCV.
6. CPI i UNRATE za październik 2025 - rekordy istnieją w silver ale value=NULL. Źródło FRED nie dostarczyło wartości, nie bug pipeline'a. Zweryfikować na stronie FRED.
7. Bieżący miesiąc (2026-03) bez danych miesięcznych - oczekiwane, jeszcze nie opublikowane.

In [0]:
%sql
DESCRIBE bronze.av_sentiment

In [0]:
%sql
WITH counts AS (
    SELECT COUNT(*) AS total_rows FROM bronze.av_sentiment
),
unique_counts AS (
    SELECT COUNT(*) AS unique_rows
    FROM (
        SELECT symbol, published_at, title
        FROM bronze.av_sentiment
        GROUP BY symbol, published_at, title
    ) t
)
SELECT 
    total_rows,
    unique_rows,
    total_rows - unique_rows AS duplicate_rows
FROM counts, unique_counts;

In [0]:
%sql
WITH counts AS (
    SELECT COUNT(*) AS total_rows FROM silver.av_sentiment
),
unique_counts AS (
    SELECT COUNT(*) AS unique_rows
    FROM (
        SELECT symbol, published_at, title
        FROM silver.av_sentiment
        GROUP BY symbol, published_at, title
    ) t
)
SELECT 
    total_rows,
    unique_rows,
    total_rows - unique_rows AS duplicate_rows
FROM counts, unique_counts;